# Assignment 3

## Mask R-CNN

### Dataset
#### NMS

##### 1. `ixs` 是什么？`ixs[0]` 是什么？
```python
ixs = np.arange(boxes.shape[0]-1, -1, -1)
```
- `ixs` 是一个**动态维护的“待处理候选框的索引列表”**。
- 初始时它包含所有框的索引，且是**从大到小**排列的：`[N-1, N-2, ..., 1, 0]`。在标准 NMS 中，这通常是因为框已经按置信度（confidence score）从高到低排过序了，所以先处理分数最高的框。
- `ixs[0]` 就是当前这一轮要拿出来的“基准框”的索引。代码中 `i = ixs[0]` 表示取出当前列表的第一个索引，作为去和其余框计算 IoU 的主体。

---

##### 2. 为什么传参是 `boxes[ixs[1:], :]` 和 `area[ixs[1:]]`？
```python
iou = compute_iou(boxes[i,:], boxes[ixs[1:],:], area[i], area[ixs[1:]])
```
- `boxes[i, :]`：当前基准框的坐标（1维数组）。
- `ixs[1:]`：**去掉第一个元素后，剩下的所有待比较框的索引**。比如 `ixs = [4,3,2,1,0]`，则 `i = 4`，`ixs[1:] = [3,2,1,0]`。
- `boxes[ixs[1:], :]` 和 `area[ixs[1:]]`：用这些索引去原数组中切片，取出**除了基准框之外，当前还需要参与比较的所有框的坐标和面积**。
- `compute_iou` 是向量化写的：第一个参数是 `1个框`，第二个参数是 `多个框`。这样一次调用就能算出基准框和剩余所有框的 IoU 数组，效率远高于 `for` 循环。

---

##### 3. `ixs = ixs[1:][iou <= threshold]` 是怎么实现“剔除”的？
这里并没有显式的 `del` 或 `remove`。它的“剔除”是靠 **NumPy 的布尔索引（Boolean Indexing）** 隐式完成的：

假设 `ixs = [4,3,2,1,0]`，当前 `i = 4`，剩余索引 `remaining = ixs[1:] = [3,2,1,0]`。
算完 IoU 后，假设得到：`iou = [0.8, 0.1, 0.6, 0.05]`，阈值 `threshold = 0.5`。

1. **生成掩码**：`iou <= threshold` → `[False, True, False, True]`
   - `False` 表示 IoU 太大（重叠严重），需要**抑制（剔除）**
   - `True` 表示 IoU 较小（可以保留到下一轮继续比较）
2. **应用掩码**：`remaining[mask]` → 取出对应 `True` 位置的索引 → `[2, 0]`
3. **重新赋值**：`ixs = [2, 0]`

**“剔除”动作发生在哪里？**
发生在**重新赋值给 `ixs` 的那一刻**。被抑制的索引（这里是 `3` 和 `1`）根本没有被放进新的 `ixs` 里。下一轮 `while` 循环开始时，`ixs` 已经变短了，那些被抑制的框自然就不会再被处理。这就是 NumPy 风格中非常高效的“逻辑删除”。

---

##### 4. `if area[i] == 0: continue` 是为什么？
```python
if area[i] == 0:
    continue
```
- **为什么会有面积为 0 的框？** 
  目标检测模型输出的框有时会出现退化情况，比如 `x1 >= x2` 或 `y1 >= y2`（宽度或高度为负数/0），经过 `area = (y2-y1)*(x2-x1)` 计算后面积就是 0 或负数。这在后处理过滤时很常见。
- **为什么要跳过？**
  1. **防除零错误**：IoU 公式分母是 `union_area = box_area + boxes_area - inter_area`。如果基准框面积为 0，且和其他框也不相交，`union_area` 可能为 0，导致 `inter_area / union_area` 触发 `DivisionByZero`。
  2. **无效框不参与抑制**：面积为 0 的框没有实际意义，不应该被加入 `pick`（保留列表），也不应该用它去抑制其他有效框。直接 `continue` 跳过它，进入下一轮 `ixs` 的循环即可。


## Depth Image to Point Cloud

In [ ]:
# P1[:, np.newaxis, :] 把 P1 变成了 (2048, 1, 3)
# P2[np.newaxis, :, :] 把 P2 变成了 (1, 2048, 3)
# 它们相减时，Numpy会自动把它们补齐成 (2048, 2048, 3)
# 意思是：P1 中的第 i 个点，减去 P2 中的第 j 个点 的 (X, Y, Z) 差值
diff = P1[:, np.newaxis, :] - P2[np.newaxis, :, :]

# 计算欧氏距离的平方 (X_diff^2 + Y_diff^2 + Z_diff^2)
# axis=-1 表示在最后一个维度 (那3个坐标轴) 上求和，得到 (2048, 2048) 的距离平方矩阵
dist_squared = np.sum(diff ** 2, axis=-1)

# 开根号，得到真实的欧氏距离 (||x - y||_2)，形状还是 (2048, 2048)
distances = np.sqrt(dist_squared)

# 对于 P1 中的每一个点（按行），找到 P2 中距离它最近的点的距离
# axis=1 表示在 P2 的那 2048 个点中挑最小的
min_distances = np.min(distances, axis=1) # 得到形状为 (2048,) 的数组

# 求平均值 (对应公式里的 1 / |S1| 乘以总和)
one_way_CD = np.mean(min_distances)